In [1]:
import os
import sys
import tqdm
project_dir = os.path.dirname(os.getcwd())
sys.path.append(project_dir)

from utils.summary import get_model_stats

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import resnet34

import warnings
warnings.filterwarnings('ignore')

In [2]:
import torch
import torch.nn as nn
from torchvision.models import resnet34

class UNet(nn.Module):
    def __init__(self, num_classes=1, pretrained=True):
        super(UNet, self).__init__()
        # Load a pretrained ResNet encoder
        self.encoder = resnet34(pretrained=pretrained)
        self.base_layers = list(self.encoder.children())

        # Encoder layers
        self.enc1 = nn.Sequential(*self.base_layers[:3])  # Conv1 + BN + ReLU
        self.enc2 = nn.Sequential(*self.base_layers[3:5])  # MaxPool + Layer1
        self.enc3 = self.base_layers[5]  # Layer2
        self.enc4 = self.base_layers[6]  # Layer3
        self.enc5 = self.base_layers[7]  # Layer4

        # Decoder with extra upsampling to restore original size
        self.up4 = self._upsample_block(512, 256)
        self.up3 = self._upsample_block(256, 128)
        self.up2 = self._upsample_block(128, 64)
        self.up1 = self._upsample_block(64, 64)
        self.up_final = nn.ConvTranspose2d(64, 64, kernel_size=3, stride=2, padding=1, output_padding=1)

        # Final output layer
        self.final_conv = nn.Conv2d(64, num_classes, kernel_size=1)

    def _upsample_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        )

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        e5 = self.enc5(e4)

        # Decoder with skip connections
        d4 = self.up4(e5) + e4
        d3 = self.up3(d4) + e3
        d2 = self.up2(d3) + e2
        d1 = self.up1(d2) + e1
        
        # Final upsampling to restore original size
        d_final = self.up_final(d1)

        # Final output
        out = self.final_conv(d_final)
        return out

# Example usage
model = UNet(num_classes=1, pretrained=True)  # Adjust num_classes for your task
input_tensor = torch.randn(1, 3, 256, 256)  # Example input
output = model(input_tensor)
print("Output shape:", output.shape)
get_model_stats(model, input_tensor.shape)

Output shape: torch.Size([1, 1, 256, 256])


Unsupported operator aten::max_pool2d encountered 1 time(s)
Unsupported operator aten::add_ encountered 16 time(s)
Unsupported operator aten::add encountered 4 time(s)
The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
encoder.avgpool, encoder.fc


{'flops': 6050349056, 'params': 24231849}

In [3]:
from data.voc import get_voc_pipeline

train_loader, val_loader, test_loader = get_voc_pipeline(batch_size=8)
sample_x, sample_y = next(iter(train_loader))
print(sample_x.shape)
print(sample_y.shape)

torch.Size([8, 3, 256, 256])
torch.Size([8, 1, 256, 256])


In [4]:
model(sample_x).shape   # IT WORKED WOOOOOOOOOOOO

torch.Size([8, 1, 256, 256])

In [5]:
import torch
import torch.nn as nn

class MIoU(nn.Module):
    def __init__(self, num_classes=21, ignore_index=255):
        super(MIoU, self).__init__()
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.reset()
        
    def reset(self):
        self.intersection = torch.zeros(self.num_classes)
        self.union = torch.zeros(self.num_classes)
        
    def forward(self, outputs, targets):
        """
        Args:
            outputs: [B, C, H, W] tensor of raw model outputs (logits)
            targets: [B, H, W] tensor of ground truth labels
        Returns:
            mean IoU score (scalar)
        """
        # Convert logits to predictions
        preds = torch.argmax(outputs, dim=1)  # [B, H, W]
        
        batch_size = outputs.size(0)
        mean_iou = 0.0
        
        for b in range(batch_size):
            current_pred = preds[b]  # [H, W]
            current_target = targets[b]  # [H, W]
            
            # Create mask for valid pixels (not ignore_index)
            valid_mask = current_target != self.ignore_index
            
            # Calculate per-class IoU
            class_iou_sum = 0.0
            valid_classes = 0
            
            for cls in range(self.num_classes):
                pred_mask = (current_pred == cls) & valid_mask
                target_mask = (current_target == cls) & valid_mask
                
                intersection = torch.logical_and(pred_mask, target_mask).sum().item()
                union = torch.logical_or(pred_mask, target_mask).sum().item()
                
                # Only count classes that appear in the ground truth
                if union > 0:
                    class_iou_sum += intersection / union
                    valid_classes += 1
            
            # Average IoU across valid classes for this image
            if valid_classes > 0:
                mean_iou += class_iou_sum / valid_classes
        
        # Average across batch
        return mean_iou / batch_size if batch_size > 0 else 0.0
    
    # Added for compatibility with previous evaluation function
    def compute(self):
        # For this implementation compute() is not needed since forward() calculates directly
        # But we keep it for interface compatibility
        return self.intersection.sum() / self.union.sum() if self.union.sum() > 0 else 0.0
    
    # Added for compatibility with previous training loop
    def update(self, outputs, targets):
        # Calculate batch IoU and accumulate stats
        with torch.no_grad():
            preds = torch.argmax(outputs, dim=1)
            
            for cls in range(self.num_classes):
                valid_mask = targets != self.ignore_index
                pred_mask = (preds == cls) & valid_mask
                target_mask = (targets == cls) & valid_mask
                
                intersection = torch.logical_and(pred_mask, target_mask).sum().item()
                union = torch.logical_or(pred_mask, target_mask).sum().item()
                
                self.intersection[cls] += intersection
                self.union[cls] += union
                

def evaluate_model(val_loader: DataLoader, model: nn.Module, criterion: nn.Module, device: str='cpu') -> list:
    """
    Evaluate the model on validation data.

    Args:
        model: The PyTorch model to evaluate.
        val_loader: DataLoader for the validation data.
        criterion: Loss function.
        device: Device to run the evaluation on ('cpu' or 'cuda').

    Returns:
        list: Collection of metrics.
    """
    model.eval()
    epoch_metrics = []
    with torch.no_grad():
        for inputs, targets in tqdm.tqdm(val_loader, desc='evaluating...', file=sys.stdout):
            inputs = inputs.to(device)
            targets = targets.squeeze(1).long().to(device)
            preds = model(inputs)
            loss = criterion(preds, targets)
            epoch_metrics.append(loss)
    return epoch_metrics

In [6]:
train_criterion = nn.CrossEntropyLoss(ignore_index=255)
val_criterion = MIoU()
val_loss = evaluate_model(val_loader, model, val_criterion, 'cpu')

evaluating...: 100%|██████████| 182/182 [02:33<00:00,  1.18it/s]


In [ ]:
import numpy as np

np.mean(val_loss)  # damn lol

np.float64(0.31667336691658005)

# Different Attempt

In [8]:
from transformers import SegformerFeatureExtractor, SegformerForSemanticSegmentation
from PIL import Image
import requests

feature_extractor = SegformerFeatureExtractor.from_pretrained("nvidia/segformer-b5-finetuned-cityscapes-1024-1024")
model = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b5-finetuned-cityscapes-1024-1024")

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

inputs = feature_extractor(images=image, return_tensors="pt")
outputs = model(**inputs)
logits = outputs.logits  # shape (batch_size, num_labels, height/4, width/4)

In [16]:
input_tensor = torch.randn(1, 3, 256, 256)  # Example input
model(input_tensor).logits.shape

torch.Size([1, 19, 64, 64])

In [12]:
logits.shape

torch.Size([1, 19, 256, 256])

In [10]:
inputs.keys()

dict_keys(['pixel_values'])

In [ ]:
get_model_stats(model, )

#### Next

In [ ]:
class ConstantWithWarmup(torch.optim.lr_scheduler._LRScheduler):
    def __init__(
        self,
        optimizer,
        num_warmup_steps: int,
    ):
        self.num_warmup_steps = num_warmup_steps
        super().__init__(optimizer)

    def get_lr(self):
        if self._step_count <= self.num_warmup_steps:
            # warmup
            scale = 1.0 - (self.num_warmup_steps - self._step_count) / self.num_warmup_steps
            lr = [base_lr * scale for base_lr in self.base_lrs]
            self.last_lr = lr
        else:
            lr = self.base_lrs
        return lr


class HyperParams:
    def __init__(self):
        self.BATCH_SIZE = 96      # not really normal batch: 64-256
        self.EMBEDDING_DIM = 1
        self.HIDDEN_DIM = 100
        self.OUTPUT_DIM = 2
        self.N_LAYERS = 1
        self.DROPOUT_RATE = 0.0
        self.LR = 0.001
        self.N_EPOCHS = 5
        self.WD = 0

hparams = HyperParams()   # LR = 0.001 (huh pretty low), decay = 0, what is eps?
WARMUP_STEPS = 200
optimizer = optim.Adam(model.parameters(), lr=hparams.LR, weight_decay=hparams.WD, eps=1e-6)
lr_scheduler = ConstantWithWarmup(optimizer, WARMUP_STEPS)


In [ ]:
class ConstantWithWarmup(torch.optim.lr_scheduler._LRScheduler):
    def __init__(
        self,
        optimizer,
        num_warmup_steps: int,
    ):
        self.num_warmup_steps = num_warmup_steps
        super().__init__(optimizer)

    def get_lr(self):
        if self._step_count <= self.num_warmup_steps:
            # warmup
            scale = 1.0 - (self.num_warmup_steps - self._step_count) / self.num_warmup_steps
            lr = [base_lr * scale for base_lr in self.base_lrs]
            self.last_lr = lr
        else:
            lr = self.base_lrs
        return lr
lr_scheduler = ConstantWithWarmup(optimizer, WARMUP_STEPS)

    
def lr_lambda(epoch):
    if epoch < warmup_epochs:
        return epoch / warmup_epochs  # Linear warmup
    return 1.0

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

In [ ]:
import torch
from torch.optim import SGD
from torch.optim.lr_scheduler import LambdaLR, MultiStepLR, SequentialLR

optimizer = SGD(model.parameters(), lr=0.1, momentum=0.9)

# Define the warmup scheduler
warmup_steps = 500
warmup_scheduler = LambdaLR(optimizer, lr_lambda=lambda step: min(1.0, step / warmup_steps))

# Define the decay scheduler (e.g., MultiStepLR for step decay)
decay_epochs = [30, 60, 90]  # Epoch milestones for decay
gamma = 0.1  # Decay factor
decay_scheduler = MultiStepLR(optimizer, milestones=decay_epochs, gamma=gamma)

# Combine the schedulers using SequentialLR
schedulers = [warmup_scheduler, decay_scheduler]
milestones = [warmup_steps]  # Switch to decay after warmup_steps
scheduler = SequentialLR(optimizer, schedulers, milestones=milestones)

In [ ]:

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Hyperparameters
batch_size = 128  # Adjust based on GPU memory
initial_lr = 0.01  # Starting learning rate
max_lr = 0.1  # Maximum learning rate after warmup
epochs = 50
warmup_epochs = 5  # Number of warmup epochs
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data preparation
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])  # Normalization
])

train_dataset = datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.CIFAR10(root='./data', train=False, transform=transform, download=True)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Model (using a ResNet50 pretrained model as an example)
from torchvision.models import resnet50
model = resnet50(pretrained=False, num_classes=10)  # Adjust number of classes
model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=initial_lr, momentum=0.9, weight_decay=5e-4)

# Learning rate scheduler with warmup
def lr_lambda(epoch):
    if epoch < warmup_epochs:
        return epoch / warmup_epochs  # Linear warmup
    return 1.0

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
# scheduler = ConstantWithWarmup(optimizer, WARMUP_STEPS)

# Training loop
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    return running_loss / len(loader)

# Evaluation loop
def evaluate(model, loader, criterion):
    model.eval()
    test_loss = 0.0
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
    accuracy = correct / len(loader.dataset)
    return test_loss / len(loader), accuracy

# Main training script
for epoch in range(epochs):
    print(f"Epoch {epoch + 1}/{epochs}")
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_accuracy = evaluate(model, test_loader, criterion)
    scheduler.step()  # Update learning rate
    print(f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}")

print("Training complete.")
